# Phase 2: Adaptive Hybrid RAG Inference Pipeline
**Objective**: Implement a high-performance, cost-efficient AI agent that dynamically routes queries between Vector (ChromaDB) and Graph (Neo4j) databases using a distilled router model.

In [1]:
!pip install -q langchain-openai langchain-community chromadb neo4j transformers openai

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from neo4j import GraphDatabase
from openai import OpenAI
from getpass import getpass

print("⚙️ All libraries imported successfully!")

In [ ]:
# Ensure OpenAI API key is set in environment variables
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

# Neo4j configuration 
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
if "NEO4J_PASSWORD" not in os.environ:
    os.environ["NEO4J_PASSWORD"] = getpass("Enter your Neo4j Password: ")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# Connect to Neo4j and OpenAI
openai_client = OpenAI() 
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) 
print("Neo4j and OpenAI clients initialized successfully!")

In [ ]:
# Loading Core Components 

# Engine A: Intent Router (Student Model)
# Loading the distilled DistilBERT model used to classify user intent with low latency.
router_path = "./models/best_router_model"
tokenizer = AutoTokenizer.from_pretrained(router_path)
router_model = AutoModelForSequenceClassification.from_pretrained(router_path)

# Engine B: ChromaDB (Vector Retrieval Engine)
# Initializing the vector store for semantic and lexical similarity search.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_db_path = "./chroma_db"
vectorstore = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
# Core Pipeline Functions

def get_routing_decision(query):
    """Classifies the query intent into one of three categories using the distilled router model."""
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = router_model(**inputs)
        predicted_class_id = outputs.logits.argmax(dim=-1).item()
    return router_model.config.id2label[predicted_class_id]

def retrieve_from_graph(query):
    """Retrieves relevant knowledge (nodes and edges) from Neo4j using LLM-based entity extraction."""

    # 1. Task GPT-4o-mini to extract core entities from the user query
    extraction_prompt = f"""
    You are an AI assistant. Extract 1 or 2 core entities from the following question.
    Return ONLY a comma-separated list of the entities. Do not write anything else.
    Example: "What is the relationship between Jesus and Mary?" -> Jesus, Mary
    Example: "Tell me about the Holy Spirit." -> Holy Spirit
    Question: {query}
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": extraction_prompt}],
        temperature=0.0 # Precision-oriented setting for consistent extraction
    )

    # "Old Testament, New Testament" -> ['Old Testament', 'New Testament']
    entities = [e.strip() for e in response.choices[0].message.content.split(",")]
    print(f"   [Debug] Core entities extracted by LLM: {entities}")

    # 2. Perform a multi-hop search in the Graph DB using the extracted entities
    context_str = ""
    with neo4j_driver.session() as session:
        for entity in entities:
            entity_clean = entity.replace(" ", "_")
            cypher_query = """
            MATCH (n)-[r]->(m)
            WHERE toLower(n.id) CONTAINS toLower($ent) OR toLower(n.id) CONTAINS toLower($ent_clean)
               OR toLower(m.id) CONTAINS toLower($ent) OR toLower(m.id) CONTAINS toLower($ent_clean)
            RETURN n.id AS source, type(r) AS relation, m.id AS target
            LIMIT 15
            """
            result = session.run(cypher_query, ent=entity, ent_clean=entity_clean)

            for record in result:
                edge = f"- {record['source']} {record['relation']} {record['target']}\n"
                if edge not in context_str: # 중복 데이터 방지
                    context_str += edge

    return context_str if context_str else "No relevant graph connections found."

def generate_final_answer(query, context, source_engine):
    """Generates a high-fidelity answer using GPT-4o based on the retrieved context."""
    system_prompt = f"""
    You are an expert Catholic theologian. Answer the user's query strictly based on the provided context.
    If the context is insufficient, say you don't know based on the Compendium.

    [Context retrieved via {source_engine}]:
    {context}
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
# Final End-to-End Orchestration Function

def adaptive_rag_pipeline(user_query):
    print(f"\n💬 User Question: '{user_query}'")

    # Step 1: Routing (Intent Classification)
    intent = get_routing_decision(user_query)
    print(f"🧭 Router Decision: Branching to [{intent}] engine.")

    context = ""
    source_engine = ""

    # Step 2: Retrieval Phase
    if intent == "Vector_Search":
        print("🔍 Performing similarity search in Vector DB (Chroma)...")
        docs = vector_retriever.invoke(user_query)
        context = "\n".join([doc.page_content for doc in docs])
        source_engine = "Vector Database"

    elif intent == "Graph_Search":
        print("🕸️ Performing multi-hop search in Graph DB (Neo4j)...")
        context = retrieve_from_graph(user_query)
        source_engine = "Knowledge Graph"

    elif intent == "Out_of_Scope":
        print("🛡️ Question is out of scope for the Catholic Compendium. Saving LLM call costs.")
        return "Sorry, but this question is out of scope for the Catholic Compendium."

    # Step 3: Answer Generation
    print("✨ GPT-4o is generating the final answer based on the retrieved context...")
    final_answer = generate_final_answer(user_query, context, source_engine)

    print("\n" + "="*50)
    print(f"🤖 Final Answer:\n{final_answer}")
    print("="*50)
    return final_answer

print("✅ Pipeline initialized and ready for inference!\n")

In [ ]:
# Inference & Testing
# Input your test queries below to see the Adaptive RAG in action!

# Case 1: Factual Query (Targeting Vector_Search engine)
adaptive_rag_pipeline("What is the explicit definition of Apostolic Tradition?")

# Case 2: Out of Scope Query (Targeting cost-saving termination logic)
adaptive_rag_pipeline("Who won the World Cup in 2022?")

# Case 3: Relational/Multi-hop Query (Targeting Graph_Search engine)
adaptive_rag_pipeline("What is the relationship between the Old Testament and the New Testament?")